In [1]:
import torch
import torch.nn as nn
import torch.optim as opt
import numpy as np
import torch_npu
from torch.utils.data import DataLoader, Dataset
#from models.simplenet import UNet
from models.unet_v3 import UNet
from models.ddpm import DDPM
# from models.unet_nocond import UNet
#from models.ddpm_nocond import DDPM_nocond as DDPM
import os
from datetime import datetime
import pandas as pd
import numpy as np
num_steps = 300
repaint_steps = 10
jump_len = 10
N = 10
n_samples = 1

device = "npu:0"
model_save_path ="/home/docker/code/Aurora_DDPM/ckpt/diffusion_ckpt_unet/ckptv1_unetv3_2/aurora_diff_best.pth"
unet = UNet(1, 1)
ddpm = DDPM(unet, num_train_steps=1000, schedule='cosine')
checkpoint = torch.load(model_save_path, map_location=device)
ddpm.load_state_dict(checkpoint['model_state_dict'],strict=False)
#ddpm.load_state_dict(checkpoint)

<All keys matched successfully>

In [2]:
from datetime import datetime
import pandas as pd
import numpy as np
from data.dataset_diff import OmniDataset
data_path = "/home/docker/data/private/AuroraData/generated_aurora_data/2005_omni_aurora/aurora_img_20050101.npy"
data_mn_all = np.load(data_path)
omni_path = "/home/docker/data/private/AuroraData/omni_real_data/omni_1min_pro/2005/omni_20050101_1min.npy"
omni_data = np.load(omni_path)
mn_time = omni_data['utc']
ssusi_data = np.load('/home/docker/data/private/AuroraData/process_ssusi/aurora_2005_ssusi.npy', allow_pickle=True)
aurora_data_ssusi = np.stack(ssusi_data['aurora_flux'], axis=0).astype(np.float32)
ssusi_timestamps = ssusi_data['utc']


solar_fields = ['Bx', 'By', 'Bz', 'V', 'P']
solar_components = []
for field in solar_fields:
    if field in omni_data.dtype.names:
        field_data = omni_data[field]
        if field_data.ndim > 1:
            if field_data.shape[1] > 0:
                field_data = field_data[:, 0]
            else:
                field_data = field_data.flatten()
        solar_components.append(field_data.astype(np.float32))
    else:
        print(f"字段 {field} 不存在，用0填充")
        solar_components.append(np.zeros(len(data), dtype=np.float32))
solar_data = np.column_stack(solar_components)
solar_data = OmniDataset(solar_data)


# print(time1)
# print(mn_time[idx_mn])

In [3]:
class normalize(nn.Module):
    def __init__(self, datas):
        super().__init__()
        self.datas = datas
        log_data = np.log1p(self.datas)
        self.log_min = float(log_data.min())
        self.log_max = float(log_data.max())
        self.log_range = max(self.log_max - self.log_min, 1e-6)
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        x = np.log1p(x)
        x = (x - self.log_min) / self.log_range
        x = np.clip(x, 0.0, 1.0)
        # return (x * 2.0 - 1.0).astype(np.float32)
        return x.astype(np.float32)
    
class denormalize(nn.Module):
    def __init__(self, datas):
        super().__init__()
        self.datas = np.array(datas, dtype=np.float32) 
        log_data = np.log1p(self.datas)
        self.log_min = float(log_data.min())
        self.log_max = float(log_data.max())
        self.log_range = max(self.log_max - self.log_min, 1e-6)
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        # aurora_01 = (x + 1.0) / 2.0
        aurora_01 = x
        aurora_log = aurora_01 * self.log_range + self.log_min
        aurora_flux = np.expm1(aurora_log)
        return aurora_flux
normalizer_real = normalize(aurora_data_ssusi)
denormalizer_real = denormalize(aurora_data_ssusi)
normalizer_mn = normalize(aurora_data_ssusi)
denormalizer_mn = denormalize(aurora_data_ssusi)

In [4]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.image import imread
import cartopy.feature as carfeat
import cartopy.io.shapereader as shpreader
from matplotlib.colors import LinearSegmentedColormap
from cartopy.feature.nightshade import Nightshade
import scipy.ndimage
import scipy.interpolate
from datetime import datetime
import os
import math
import aacgmv2
from datetime import timedelta
def convert_datetime64_to_datetime(dt64):
    """将 numpy.datetime64 转换为 datetime.datetime"""
    if dt64 is None:
        return None
    if isinstance(dt64, datetime):
        return dt64
    import pandas as pd
    return pd.Timestamp(dt64).to_pydatetime()

def plot_real(timestamp, energy_flux, save_path):
    # 1. 准备你的数据
    # 假设你有网格数据：energy_flux[mlat_bins, mlt_bins]
    timestamp = convert_datetime64_to_datetime(timestamp)
    mlat = np.linspace(50, 90, 80)  # 纬度网格
    mlt = np.linspace(0, 24, 96)    # 地方时网格
    MLAT, MLT = np.meshgrid(mlat, mlt)  # 创建网格

    # 2. 转换为绘图坐标（极坐标）
    # 将MLT转换为角度（弧度）
    # 减去π/2使0 MLT在底部（-90度）
    theta = (MLT / 24.0) * 2 * np.pi - np.pi/2

    # 将MLAT转换为半径
    # 纬度越高（接近90°），半径越小
    # 我们想要从中心（90°）到边缘（50°）的半径从0到1
    r = (90 - MLAT) / 40.0  # 因为90-50=40度范围

    x = r * np.cos(theta)
    y = r * np.sin(theta)

    # 3. 创建极坐标投影的地图
    # fig = plt.figure(figsize=(8, 8))
    # # 使用PlateCarree投影的极坐标视图
    # ax = plt.axes(projection=ccrs.NorthPolarStereo())
    # ax.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())

    fig, ax = plt.subplots(figsize=(6, 5), subplot_kw={'projection': 'polar'})

    # 4. 绘制填色图
    # 4. 自定义颜色映射 - 黑->蓝->绿->红->灰
    # 这是论文中常用的色彩映射
    colors = [
        (0, 0, 0),          # 黑色 (最低值)
        (0, 0, 0.5),        # 深蓝
        (0, 0, 0.8),        # 蓝色
        (0, 0.5, 1),        # 天蓝
        (0, 1, 1),          # 青色
        (0.5, 1, 0.5),      # 青绿
        (1, 1, 0),          # 黄色
        (1, 0.5, 0),        # 橙色
        (1, 0, 0),          # 红色
        (0.8, 0.8, 0.8)     # 灰色 (最高值)
    ]
    custom_cmap = LinearSegmentedColormap.from_list('aurora_cmap', colors, N=256)


    # ==================== 5. 设置极坐标图的属性 ====================
    ax.set_theta_zero_location('S')  # 0度在底部（对应0 MLT）
    ax.set_theta_direction(1)       # 角度顺时针增加（这是空间物理的标准）
    ax.set_ylim(0, 1)                # 半径范围从0到1

    # 隐藏默认的径向标签（我们要用纬度标签）
    ax.set_yticklabels([])

    # 设置角度刻度为地方时
    hour_ticks = np.arange(0, 24, 1)
    angle_ticks = (hour_ticks / 24.0) * 360  # 转换为角度
    ax.set_xticks(np.deg2rad(angle_ticks))

    # 设置刻度标签 - 只显示0,6,12,18，其他为空
    mlt_labels = []
    for hour in hour_ticks:
        if hour in [0, 6, 12, 18]:
            mlt_labels.append(str(hour))
        else:
            mlt_labels.append('')
    ax.set_xticklabels(mlt_labels, fontsize=10)

    # 7. 设置8个径向网格线（纬度圈）的位置
    # 从50°到85°，每5°一个，共8个圈
    lat_circles = [50, 55, 60, 65, 70, 75, 80, 85]
    radial_ticks = [(90 - lat) / 40.0 for lat in lat_circles]  # 转换为半径

    # 设置径向网格线的位置和标签
    ax.set_rticks(radial_ticks)
    ax.set_yticklabels([f'{lat}°' for lat in lat_circles], 
                    fontsize=5, color='white')
    time_str = timestamp.strftime("%Y-%m-%d %H:%M UT")
    c = ax.pcolormesh(theta, r, energy_flux.T,
                    cmap=custom_cmap, shading='auto',vmin=0, vmax=5)
    title = "Auroral Energy Flux Map"
    fig.text(0.5, 0.95, f'{title} - {time_str}',
                color='black', fontsize=8, ha='center', weight='bold')
    cbar_ax = fig.add_axes([0.85, 0.25, 0.03, 0.5])  # [left, bottom, width, height]
    plt.colorbar(c, cax=cbar_ax, pad=0.1, label='ergs cm⁻² s⁻¹')

    plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]
    plt.savefig(save_path, dpi=300, facecolor='white')
    #plt.show()

def plot(timestamp, repaired_flux, save_path):
    timestamp = convert_datetime64_to_datetime(timestamp)
    lat_coords = np.linspace(50, 90, 80)
    mlt_coords = np.linspace(0.0, 24.0, 96)
    mltN, mlatN = np.meshgrid(mlt_coords, lat_coords)
    mlonN_1D_small = aacgmv2.convert_mlt(mltN[0], timestamp, m2a=True)
    mlonN_1D = np.tile(mlonN_1D_small, mlatN.shape[0])
    mlatN_1D = np.squeeze(mlatN.reshape(np.size(mltN), 1))

    (glatN_1D, glonN_1D, galtN) = aacgmv2.convert_latlon_arr(mlatN_1D, mlonN_1D, 100, timestamp,
                                                                        method_code="A2G")

    # 插值到世界地图网格
    geo_2D = np.vstack((glatN_1D, glonN_1D)).T
    fluxN_1D = repaired_flux.reshape(7680, 1)

    # 创建世界地图网格 - 使用更小的网格
    h, w = 512, 1024
    wx, wy = np.mgrid[-90:90:180 / h, -180:180:360 / w]

    # 线性插值
    aimg = np.squeeze(scipy.interpolate.griddata(geo_2D, fluxN_1D, (wx, wy), method='linear', fill_value=0))

    # 高斯平滑处理
    aimg = scipy.ndimage.gaussian_filter(aimg, sigma=(2, 3), mode='wrap')

    aimg = aimg.astype(np.float32)

    colors = [
            (0.0, 0.0, 0.0),
            (0.0, 0.2, 0.0),
            (0.0, 0.5, 0.0),
            (0.0, 0.8, 0.0),
            (0.5, 1.0, 0.0),
            (1.0, 1.0, 0.0),
            (1.0, 0.6, 0.0),
            (1.0, 0.3, 0.0),
            (1.0, 0.0, 0.0),
        ]
    cmap = LinearSegmentedColormap.from_list('aurora', colors, N=256)
    fig = plt.figure(figsize=(12, 10), dpi=150)
    fig.set_facecolor('white')

    ax = fig.add_subplot(1, 1, 1,
                            projection=ccrs.Orthographic(116.2,90))
                            #position=[0, 0, 1, 1])
                            #position=[0.3, 0.1, 0.45, 0.45])
    
    background_img = '/home/docker/data/private/AuroraData/background_img/natural-earth-1_large2048px.png'
    map_img = imread(background_img)
    ax.imshow(map_img, origin='upper', transform=ccrs.PlateCarree(),
                extent=[-180, 180, -90, 90], zorder=0)


    gl = ax.gridlines(linestyle='solid', alpha=0.5, color='white')
    gl.n_steps = 100
    gl.xlocator = matplotlib.ticker.FixedLocator(np.arange(-180, 190, 45))
    gl.ylocator = matplotlib.ticker.FixedLocator(np.arange(-90, 100, 10))



    ax.coastlines('10m', color='white', alpha=0.4)

    ax.add_feature(Nightshade(timestamp))
    img = ax.imshow(aimg,
                    vmin=0,
                    vmax=5,
                    transform=ccrs.PlateCarree(),
                    extent=[-180, 180, -90, 90],
                    origin='lower',
                    zorder=3,
                    alpha=0.8,
                    cmap=cmap)
    ax.set_facecolor('white')

    if timestamp is not None:
        time_str = timestamp.strftime("%Y-%m-%d %H:%M UT")
    else:
        time_str = "Unknown Time"
    title = "Aurora ForecastNet Repaired Aurora"
    fig.text(0.5, 0.95, f'{title} - {time_str}',
                color='white', fontsize=18, ha='center', weight='bold')

    cbaxes = fig.add_axes([0.3, 0.07, 0.4, 0.02])
    cbar = plt.colorbar(img, cax=cbaxes, orientation='horizontal')
    cbar.set_alpha(1)
    cbar.ax.tick_params(labelsize=15, colors='white')
    cbar.set_label(r'aurora flux $\mathrm{erg\/cm^{-2}\/s^{-1}}$',
                    color='white', fontsize=16)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, facecolor='black')

In [ ]:
def creat_gif(image_dir, duration=500, loop=0, image_nums=20):
    from PIL import Image
    pil_images = []
    for i in range(image_nums):
        img_path = os.path.join(image_dir, f'repaired_aurora_{i}.png')
        pil_img = Image.open(img_path).copy()
        pil_images.append(pil_img)
    save_path = os.path.join(image_dir, 'repaired_aurora_polar_2.gif')
    pil_images[0].save(save_path, save_all=True, append_images=pil_images[1:],
                      duration=duration, loop=loop, optimize=False)

In [5]:
def train(input_data, mask, solar_point ,time1, img_path):
    ddpm.eval()
    ddpm.to(device)
    with torch.no_grad():
        # 使用ddpm.sample进行修补
        repaired = ddpm.sample(
            input_data,
            mask,
            solar_point,
            num_inference_steps=num_steps,
            n_sample=n_samples,
            j=jump_len,
            r=repaint_steps,
        )
    repaired_np = repaired.cpu().numpy().squeeze()
    repaired_flux = denormalizer_real(repaired_np)
    plot(time1, repaired_flux, img_path)
    #return repaired_flux
    #plot_real(time1, repaired_flux, img_path)

In [6]:
img_dir = "/home/docker/code/Aurora_DDPM/reasult/dmsp_res/new_res/repaired_aurora_imgs_ssusi_unetv3_ckptv1_2/"
os.makedirs(img_dir, exist_ok=True)

for i in range(0, 15):
    ssusi1= aurora_data_ssusi[i]
    time1 = ssusi_timestamps[i].astype('datetime64[s]')
    time1 = pd.Timestamp(time1).to_pydatetime()
    base_time = datetime.fromisoformat("2005-01-01T00:00:00")
    delta = time1 - base_time
    idx_mn = int(delta.total_seconds() // 60 ) # Assuming 5-minute intervals
    mn_data = data_mn_all[idx_mn]
    solar_point = solar_data[idx_mn]
    
    print("ssusi:",time1)
    print("omni_time:",mn_time[idx_mn])
    
    mask = np.ones_like(ssusi1)
    real_close_to_zero = ssusi1 < 1
    sim_has_value = mn_data > 0
    need_repair = real_close_to_zero & sim_has_value
    mask[need_repair] = 0.0
    mask = np.expand_dims(mask, axis=(0, 1))   
    
    ssusi1 = normalizer_real(ssusi1)
    ssusi1 = np.expand_dims(ssusi1, axis=(0,1))
    input_data = ssusi1.copy()
    input_data = torch.tensor(input_data).float().to(device)
    mask = torch.tensor(mask).float().to(device)
    solar_point = solar_point.unsqueeze(0).to(device)
    print(solar_point.shape)
    img_path = os.path.join(img_dir, f'repaired_ssusi_{i}.png')
    train(input_data, mask, solar_point ,time1, img_path)

ssusi: 2005-01-01 02:00:04
omni_time: 2005-01-01T02:00:00.000000000
torch.Size([1, 5])


/tmp/ipykernel_2981044/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


ssusi: 2005-01-01 03:42:01
omni_time: 2005-01-01T03:42:00.000000000
torch.Size([1, 5])


/tmp/ipykernel_2981044/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


ssusi: 2005-01-01 05:23:57
omni_time: 2005-01-01T05:23:00.000000000
torch.Size([1, 5])


...

/tmp/ipykernel_2981044/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


ssusi: 2005-01-02 01:47:09
omni_time: 2005-01-02T01:47:00.000000000
torch.Size([1, 5])


/tmp/ipykernel_2981044/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


ssusi: 2005-01-02 03:29:05
omni_time: 2005-01-02T03:29:00.000000000
torch.Size([1, 5])


/tmp/ipykernel_2981044/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
